# Shear Center Calculation — S7055 Wing Section

This notebook calculates the shear-center location of the closed S7055 airfoil section using:

1. Thin-walled skin idealised as boom areas.
2. Added stringer/boom areas.
3. Final centroid of the idealised cross-section.
4. Centroidal second moments of area.
5. Basic shear flow due to a vertical shear force.
6. Constant redundant shear flow for a single-cell closed section.
7. Torque about the centroid.
8. Shear-center location.

The calculation uses the existing `week1_sd.py` and `week2a_sd.py` data conventions, but performs the shear-center calculation consistently about the **final centroid**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rc('font', size=14)
np.set_printoptions(precision=5, suppress=True)

# ---------------------------------------------------------------------
# INPUTS
# ---------------------------------------------------------------------

airfoil_file = "s7055.dat"

# Wing / section data
chord = 0.265
skin_thickness = 0.5e-3

# Stringer geometry
stringer_thickness = 0.5e-3
stringer_length = 20e-3

# Number of stringers to add.
# Change this to match your section.
num_stringers = 0

# Stringer positions as normalized chordwise coordinates.
# Example: np.linspace(0.05, 0.95, 8)
# If num_stringers > 0, replace this with your actual positions.
stringer_x_norm = np.array([])

# Vertical shear force used only to obtain the shear-center moment.
# The resulting shear-center location is independent of its magnitude.
V = 1.0  # N


In [ ]:

# ---------------------------------------------------------------------
# LOAD AIRFOIL
# ---------------------------------------------------------------------

airfoil = np.loadtxt(airfoil_file) * chord

# Selig coordinates contain a duplicate trailing-edge point.
# Remove the final duplicate if it is coincident with the first point.
if np.allclose(airfoil[0], airfoil[-1]):
    airfoil = airfoil[:-1]

x = airfoil[:, 0]
y = airfoil[:, 1]

n = len(x)

# Closed panels
x_next = np.roll(x, -1)
y_next = np.roll(y, -1)

dx = x_next - x
dy = y_next - y
ds = np.hypot(dx, dy)

# Panel midpoints
x_mid = 0.5 * (x + x_next)
y_mid = 0.5 * (y + y_next)


In [ ]:

# ---------------------------------------------------------------------
# 1. SKIN AREA IDEALISATION
# ---------------------------------------------------------------------

A_skin_panel = skin_thickness * ds
A_skin = np.sum(A_skin_panel)

x_centroid_skin = np.sum(A_skin_panel * x_mid) / A_skin
y_centroid_skin = np.sum(A_skin_panel * y_mid) / A_skin

print(f"Skin area       = {A_skin:.6e} m^2")
print(f"Skin centroid x = {x_centroid_skin:.6e} m")
print(f"Skin centroid y = {y_centroid_skin:.6e} m")

# ---------------------------------------------------------------------
# 2. BOOM AREAS
#
# Use the standard idealisation:
#
# B_i = t/6 * [
#       l_(i-1) (2 + y_(i-1)/y_i)
#     + l_i     (2 + y_(i+1)/y_i)
#     ]
#
# Here the boom formula is evaluated using the skin centroidal
# y-coordinate. The final centroid is then calculated from the
# complete idealised section.
#
# Points extremely close to the reference axis make this formula
# singular. For a robust shear-center calculation, use panel-area
# discretisation instead when such points occur.
# ---------------------------------------------------------------------

yc_skin = y - y_centroid_skin

# Check for points too close to the reference axis.
if np.any(np.abs(yc_skin) < 1e-10):
    raise ValueError(
        "A boom lies essentially on the skin centroidal axis. "
        "The standard boom-area formula becomes singular. "
        "Use a different boom/reference discretisation."
    )

l_prev = np.roll(ds, 1)
l_next = ds

y_prev = np.roll(yc_skin, 1)
y_next = np.roll(yc_skin, -1)

B_skin = (
    skin_thickness / 6.0
    * (
        l_prev * (2.0 + y_prev / yc_skin)
        + l_next * (2.0 + y_next / yc_skin)
    )
)


In [ ]:

# ---------------------------------------------------------------------
# 3. ADD STRINGER BOOM AREAS
# ---------------------------------------------------------------------

B_stringer = np.zeros(n)

if num_stringers > 0:
    if len(stringer_x_norm) != num_stringers:
        raise ValueError(
            "num_stringers must equal len(stringer_x_norm)."
        )

    # Interpolate upper/lower surface positions.
    # For this simple setup, place each stringer on the upper surface.
    # Replace with your actual stringer coordinates if required.
    #
    # Find the airfoil point nearest to each requested x location.
    for xs in stringer_x_norm * chord:
        idx = np.argmin(np.abs(x - xs))
        B_stringer[idx] += stringer_thickness * stringer_length

B_total = B_skin + B_stringer

# ---------------------------------------------------------------------
# 4. FINAL CENTROID
# ---------------------------------------------------------------------

A_total = np.sum(B_total)

x_bar = np.sum(B_total * x) / A_total
y_bar = np.sum(B_total * y) / A_total

x_c = x - x_bar
y_c = y - y_bar

print()
print(f"Total idealised area = {A_total:.6e} m^2")
print(f"Final centroid x     = {x_bar:.6e} m")
print(f"Final centroid y     = {y_bar:.6e} m")


In [ ]:

# ---------------------------------------------------------------------
# 5. SECOND MOMENTS OF AREA
# ---------------------------------------------------------------------

Ixx = np.sum(B_total * y_c**2)
Iyy = np.sum(B_total * x_c**2)
Ixy = np.sum(B_total * x_c * y_c)

D = Ixx * Iyy - Ixy**2

print()
print(f"Ixx = {Ixx:.6e} m^4")
print(f"Iyy = {Iyy:.6e} m^4")
print(f"Ixy = {Ixy:.6e} m^4")
print(f"D   = {D:.6e} m^8")

# ---------------------------------------------------------------------
# 6. BASIC SHEAR FLOW
#
# For a vertical shear force V:
#
# q_b = V/D * (Ixy*x - Ixx*y) integrated as a cumulative
# boom contribution.
#
# The first boom is taken as the cut/reference point.
#
# The panel flow is reconstructed by averaging adjacent nodal flows.
# ---------------------------------------------------------------------

# Contribution at each boom.
dq_b = (
    V / D
    * (
        Ixy * x_c
        - Ixx * y_c
    )
    * B_total
)

# Cumulative basic shear flow.
q_b = np.zeros(n)

for i in range(1, n):
    q_b[i] = q_b[i - 1] + dq_b[i - 1]


In [ ]:

# ---------------------------------------------------------------------
# 7. REDUNDANT CONSTANT SHEAR FLOW
#
# For a single-cell closed section:
#
# integral(q ds) = 0
#
# Therefore:
#
# q0 = - integral(q_b ds) / integral(ds)
# ---------------------------------------------------------------------

q_b_panel = 0.5 * (q_b + np.roll(q_b, -1))

q0 = -np.sum(q_b_panel * ds) / np.sum(ds)

q_panel = q_b_panel + q0

print()
print(f"Basic-flow closure correction q0 = {q0:.6e} N/m")


In [ ]:

# ---------------------------------------------------------------------
# 8. TORQUE ABOUT THE FINAL CENTROID
#
# For each straight panel:
#
# dT = q * (x dy - y dx)
#
# For a straight panel:
#
# integral(x dy - y dx)
#     = x_i*y_(i+1) - y_i*x_(i+1)
# ---------------------------------------------------------------------

panel_moment_arm = (
    x_c * np.roll(y_c, -1)
    - y_c * np.roll(x_c, -1)
)

T_shear_flow = np.sum(q_panel * panel_moment_arm)

# Since V = 1 N:
x_sc = T_shear_flow / V

print()
print(f"Torque about centroid = {T_shear_flow:.6e} N m")
print(f"Shear-center x offset = {x_sc:.6e} m")
print(f"Shear-center x/c      = {x_sc / chord:.6f}")

# Absolute coordinate from LE
x_sc_LE = x_bar + x_sc

print(f"Shear-center x from LE = {x_sc_LE:.6e} m")
print(f"Shear-center x/c       = {x_sc_LE / chord:.6f}")

# ---------------------------------------------------------------------
# 9. PLOT
# ---------------------------------------------------------------------

plt.figure(figsize=(10, 5))
plt.plot(x, y, '-k', label='S7055 skin')
plt.scatter(x_bar, y_bar, s=70, label='Centroid')
plt.scatter(x_sc_LE, y_bar, s=90, marker='x', label='Shear center')

plt.axhline(y_bar, linestyle='--', linewidth=1)
plt.axvline(x_sc_LE, linestyle='--', linewidth=1)

plt.axis('equal')
plt.xlabel('x (m)')
plt.ylabel('y (m)')
plt.title('S7055 Shear Center')
plt.grid()
plt.legend()
plt.show()

# ---------------------------------------------------------------------
# 10. SHEAR-FLOW PLOT
# ---------------------------------------------------------------------

plt.figure(figsize=(10, 5))
plt.plot(x_mid, q_panel, '-')
plt.xlabel('x (m)')
plt.ylabel('Shear flow q (N/m)')
plt.title('Closed-Section Shear Flow')
plt.grid()
plt.show()


## Important note on stringers

Set `num_stringers` and `stringer_x_norm` to your actual stringer arrangement.

The current notebook defaults to **no stringers** so that the basic S7055 closed-section shear-center calculation can be checked first.

For example:

```python
num_stringers = 4
stringer_x_norm = np.array([0.1, 0.3, 0.7, 0.9])
```

The present code places those stringer boom areas at the nearest airfoil nodes. For your actual C-spar/stringer configuration, the stringer coordinates should instead be entered explicitly.
